# Figures

Run notebooks 01 and 03 first.

| Figure | Content | Reads |
|---|---|---|
| Fig 1 | Answers to the ethical questions, by group | `item_distributions.csv` |
| Fig 2 | Parties held accountable, by group, with 95% CIs | `key_percentages.csv` |
| Fig 3 | Correlations between views | `correlations.csv` |

**Writes to `results/figures/`:** `fig1_opinion_answers`, `fig2_accountability`, `fig3_correlations` 

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch

RESULTS = Path("../results")
FIG_DIR = RESULTS / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

MM = 1 / 25.4
FULL_W = 170 * MM
plt.rcParams.update({
    "font.family": "Arial", "font.size": 7, "axes.titlesize": 7.5, "axes.labelsize": 7,
    "xtick.labelsize": 6.5, "ytick.labelsize": 6.5, "legend.fontsize": 6.5,
    "pdf.fonttype": 42, "ps.fonttype": 42, "savefig.dpi": 300,
    "axes.edgecolor": "#c3c2b7", "axes.linewidth": 0.6, "xtick.color": "#52514e", "ytick.color": "#52514e",
    "xtick.major.width": 0.6, "ytick.major.width": 0.6, "text.color": "#0b0b0b", "axes.labelcolor": "#0b0b0b",
})

# Colour-blind safe palette; group colours also carry a hatch pattern for greyscale printing.
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e1e0d9"
GROUP_COLOURS = {"Patient": "#2a78d6", "Healthcare professional": "#eb6834", "AI developer": "#1baf7a"}
GROUP_HATCH = {"Patient": "", "Healthcare professional": "////", "AI developer": "\\\\\\\\"}
ALL_COLOUR = "#52514e"
BLUE = {"light": "#86b6ef", "mid": "#2a78d6", "dark": "#104281"}
RED = {"light": "#f1a3a2", "mid": "#e34948", "dark": "#a52a29"}
NEUTRAL = "#d4d3cd"

GROUPS = ["Patient", "Healthcare professional", "AI developer"]
ROWS = GROUPS + ["Total"]
GROUP_LABEL = {"Patient": "Patients", "Healthcare professional": "Healthcare professionals",
               "AI developer": "AI developers", "Total": "All participants"}


def save(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(FIG_DIR / (name + "." + ext), bbox_inches="tight", pad_inches=0.02)
    plt.close(fig)
    print("Saved", name)

In [2]:
dist = pd.read_csv(RESULTS / "item_distributions.csv")
key = pd.read_csv(RESULTS / "key_percentages.csv")
corr = pd.read_csv(RESULTS / "correlations.csv")
N = key.drop_duplicates("group").set_index("group")["n"].to_dict()
N

{'Patient': 48,
 'Healthcare professional': 58,
 'AI developer': 147,
 'Total': 253}

## Fig 1: answers to the ethical questions, by group

Diverging bars: answers on the negative side of each scale go left of zero and answers on the positive side go right. Neutral, Not sure and Little trust are centred on zero. The numbers at each end give the total percentage on that side.

In [3]:
QUESTIONS = [  # (block title, questions, answer order, colours, side: -1 left, 0 centre, 1 right, legend labels)
    ("Agreement scale", [
        ("privacy_protection", "Q8. AI systems adequately protect patient data and privacy"),
        ("autonomy_undermined", "Q13. AI could undermine the autonomy of patients and providers"),
    ], ["Strongly disagree", "Disagree", "Neutral", "Agree", "Strongly agree"],
       [RED["dark"], RED["light"], NEUTRAL, BLUE["light"], BLUE["dark"]], [-1, -1, 0, 1, 1],
       ["Strongly disagree", "Disagree", "Neutral", "Agree", "Strongly agree"]),
    ("Concern, importance and confidence scales", [
        ("bias_concern", "Q7. Concern: AI perpetuating biases in clinical decisions"),
        ("disparities_concern", "Q16. Concern: AI increasing disparities in access to care"),
        ("explanation_importance", "Q9. Importance of clear explanations of AI decisions"),
        ("guidelines_confidence", "Q14. Confidence that current ethical guidelines are sufficient"),
    ], None, [RED["mid"], NEUTRAL, BLUE["light"], BLUE["mid"], BLUE["dark"]], [-1, 0, 1, 1, 1],
       ["Not (concerned / important / confident)", "Neutral", "Somewhat", "Concerned / Important / Confident", "Extremely"]),
    ("Trust", [
        ("trust", "Q12. Trust in AI to make accurate and fair decisions, compared with professionals"),
    ], ["No trust", "Little trust", "Complete trust"], [RED["mid"], BLUE["light"], BLUE["dark"]], [-1, 0, 1],
       ["No trust", "Little trust", "Complete trust"]),
    ("Yes / No / Not sure", [
        ("inform_patients", "Q10. Patients should always be informed when AI is used"),
        ("no_human_life_death", "Q15. Comfortable with AI making life-and-death decisions without a human"),
    ], ["No", "Not sure", "Yes"], [RED["mid"], NEUTRAL, BLUE["mid"]], [-1, 0, 1], ["No", "Not sure", "Yes"]),
]

n_items = sum(len(q[1]) for q in QUESTIONS)
fig = plt.figure(figsize=(FULL_W, 218 * MM))
gs = fig.add_gridspec(n_items + len(QUESTIONS), 1,
                      height_ratios=sum([[0.62] + [1.0] * len(q[1]) for q in QUESTIONS], []), hspace=0.6)
row = 0
for block_title, items, order, colours, sides, legend_labels in QUESTIONS:
    lax = fig.add_subplot(gs[row]); lax.axis("off"); row += 1
    lax.legend(handles=[Patch(facecolor=c, edgecolor="white", label=l) for c, l in zip(colours, legend_labels)],
               loc="center left", ncol=len(colours), frameon=False, handlelength=1.2, columnspacing=1.0,
               bbox_to_anchor=(-0.005, 0.75), title=block_title, title_fontproperties={"weight": "bold", "size": 7},
               alignment="left")
    for var, title in items:
        ax = fig.add_subplot(gs[row]); row += 1
        answers = list(dist.loc[dist["variable"] == var, "answer"])
        cats = order or answers
        assert set(cats) == set(answers), var
        for i, r in enumerate(ROWS):
            pct = [float(dist.loc[(dist["variable"] == var) & (dist["answer"] == c), r + "_pct"].iloc[0]) for c in cats]
            neg = sum(p for p, s in zip(pct, sides) if s < 0)
            mid = sum(p for p, s in zip(pct, sides) if s == 0)
            pos = sum(p for p, s in zip(pct, sides) if s > 0)
            left = -(neg + mid / 2)
            y = len(ROWS) - 1 - i
            for p, c in zip(pct, colours):
                ax.barh(y, p, left=left, height=0.72, color=c, edgecolor="white", linewidth=0.6)
                left += p
            ax.text(-(neg + mid / 2) - 1.5, y, format(neg, ".0f") + "%", ha="right", va="center", fontsize=5.8, color=INK2)
            ax.text(pos + mid / 2 + 1.5, y, format(pos, ".0f") + "%", ha="left", va="center", fontsize=5.8, color=INK2)
        ax.axvline(0, color=INK2, linewidth=0.6)
        ax.set_xlim(-100, 100)
        ax.set_yticks(range(len(ROWS)))
        ax.set_yticklabels([GROUP_LABEL[r] + " (" + str(N[r]) + ")" for r in reversed(ROWS)])
        ax.set_title(title, loc="left", fontsize=7, pad=2)
        for s in ["top", "right", "left"]:
            ax.spines[s].set_visible(False)
        ax.tick_params(axis="y", length=0)
        is_last = row == n_items + len(QUESTIONS)
        ax.set_xticks([-100, -50, 0, 50, 100])
        ax.set_xticklabels(["100", "50", "0", "50", "100"] if is_last else [])
        if is_last:
            ax.set_xlabel("Percentage of participants (left: negative side of scale; right: positive side)")
save(fig, "fig1_opinion_answers")

Saved fig1_opinion_answers


## Fig 2: parties held accountable, by group (95% Wilson confidence intervals)

In [4]:
ACC = [("Q11 accountable: healthcare professional", "Healthcare professional"),
       ("Q11 accountable: AI developers", "AI developers"),
       ("Q11 accountable: hospital or clinic", "Hospital or clinic")]
fig, ax = plt.subplots(figsize=(68 * MM, 72 * MM))
bar_h = 0.19
for j, (measure, label) in enumerate(ACC):
    for i, r in enumerate(ROWS):
        k = key[(key["measure"] == measure) & (key["group"] == r)].iloc[0]
        y = (len(ACC) - 1 - j) + (1.5 - i) * bar_h
        ax.barh(y, k["pct"], height=bar_h * 0.88, color=GROUP_COLOURS.get(r, ALL_COLOUR), hatch=GROUP_HATCH.get(r, ""),
                edgecolor="white", linewidth=0.4)
        ax.errorbar(k["pct"], y, xerr=[[k["pct"] - k["ci_low"]], [k["ci_high"] - k["pct"]]], fmt="none",
                    ecolor=INK, elinewidth=0.6, capsize=1.5, capthick=0.6)
ax.set_yticks(range(len(ACC)))
ax.set_yticklabels([a[1] for a in reversed(ACC)])
ax.set_xlim(0, 100)
ax.set_xlabel("Participants choosing this party (%), with 95% CI")
ax.set_ylabel("Party held accountable")
ax.xaxis.grid(True, color=GRID, linewidth=0.5); ax.set_axisbelow(True)
for s in ["top", "right", "left"]:
    ax.spines[s].set_visible(False)
ax.tick_params(axis="y", length=0)
ax.legend(handles=[Patch(facecolor=GROUP_COLOURS.get(r, ALL_COLOUR), hatch=GROUP_HATCH.get(r, ""), edgecolor="white",
                         label=GROUP_LABEL[r] + " (" + str(N[r]) + ")") for r in ROWS],
          loc="upper center", bbox_to_anchor=(0.35, -0.2), ncol=2, frameon=False, handlelength=1.6,
          columnspacing=0.8, fontsize=6)
save(fig, "fig2_accountability")

Saved fig2_accountability


## Fig 3: correlations between views (exploratory)

Spearman correlations, lower triangle. An asterisk marks pairs with p < 0.05 after Benjamini-Hochberg correction.

In [5]:
ORDER = ["Familiarity with AI", "Privacy is protected (agree)", "Trust in AI", "Guidelines are enough (confidence)",
         "Explanations important", "Autonomy could be undermined (agree)", "Bias concern", "Disparities concern"]
SHORT = {"Privacy is protected (agree)": "Privacy is protected", "Guidelines are enough (confidence)": "Guidelines are enough",
         "Autonomy could be undermined (agree)": "Autonomy at risk"}
k = len(ORDER)
rho = np.full((k, k), np.nan)
sig = np.zeros((k, k), bool)
for _, r in corr.iterrows():
    i, j = ORDER.index(r["var_1"]), ORDER.index(r["var_2"])
    rho[max(i, j), min(i, j)] = r["rho"]
    sig[max(i, j), min(i, j)] = r["after_bh"]

cmap = LinearSegmentedColormap.from_list("div", [RED["dark"], RED["mid"], "#f0efec", BLUE["mid"], BLUE["dark"]])
fig, ax = plt.subplots(figsize=(70 * MM, 64 * MM))
im = ax.imshow(np.ma.masked_invalid(rho), cmap=cmap, vmin=-0.5, vmax=0.5)
for a in range(k):
    for b in range(k):
        if not np.isnan(rho[a, b]):
            v = rho[a, b]
            ax.text(b, a, format(v, ".2f").replace("0.", ".") + ("*" if sig[a, b] else ""), ha="center", va="center",
                    fontsize=5, color="white" if abs(v) >= 0.3 else INK)
labels = [SHORT.get(o, o) for o in ORDER]
ax.set_xticks(range(k)); ax.set_yticks(range(k))
ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=5.5)
ax.set_yticklabels(labels, fontsize=5.5)
ax.set_xlim(-0.5, k - 1.5); ax.set_ylim(k - 0.5, 0.5)
for s in ax.spines.values():
    s.set_visible(False)
ax.tick_params(length=0)
cb = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02, ticks=[-0.5, -0.25, 0, 0.25, 0.5])
cb.outline.set_visible(False); cb.ax.tick_params(labelsize=5.5, length=0)
cb.set_label("Spearman rho", fontsize=6)
save(fig, "fig3_correlations")

Saved fig3_correlations
